# Installations

In [4]:
%pip install -Uq "unstructured[all-docs]" pillow lxml pillow
%pip install -Uq chromadb tiktoken
%pip install -Uq langchain langchain-community langchain-openai langchain-groq
%pip install -Uq python_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.2/542.2 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [5]:
pip install rank_bm25

In [10]:
!pip install -U --force-reinstall Pillow

  Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)
  Attempting uninstall: Pillow
    Found existing installation: pillow 10.4.0
    Uninstalling pillow-10.4.0:
      Successfully uninstalled pillow-10.4.0


In [7]:
!pip install langchain-google-genai
!pip install -U langchain-huggingface
!apt-get install -y poppler-utils
!pip install open-clip-torch



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 960.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.13 [186 kB]
Fetched 186 kB in 1s (129 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd

# Importing Main Libraries

In [1]:
from unstructured.partition.pdf import partition_pdf
import numpy as np
from PIL import Image
import torch
from transformers import CLIPProcessor , CLIPModel
from langchain_google_genai import ChatGoogleGenerativeAI
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from chromadb.utils.data_loaders import ImageLoader

In [2]:
from pypdf import PdfReader, PdfWriter

def extract_pages_to_new_pdf(input_pdf_path, output_pdf_path, start_page, end_page):
    reader = PdfReader(input_pdf_path)
    writer = PdfWriter()

    for page_num in range(start_page - 1, end_page):
        writer.add_page(reader.pages[page_num])


    with open(output_pdf_path, "wb") as output_file:
        writer.write(output_file)

    print(f"Successfully extracted pages {start_page} to {end_page} into {output_pdf_path}!")

extract_pages_to_new_pdf(
    input_pdf_path="1830_Technical_Description.pdf",
    output_pdf_path="extracted_nokia_pages.pdf",
    start_page=47,
    end_page=166
)

Successfully extracted pages 47 to 166 into extracted_nokia_pages.pdf!


# Split PDF into Chunks

In [3]:
def PDF_Splitter(path):
  chunks=partition_pdf(
      filename=path,
      infer_table_structure=True,
      chunking_strategy="by_title",
      max_characters=2000,



  )
  return chunks

# Separating the extracyed into Text & Tables

In [4]:
def Extract_data(chunks):
  Images=[]
  Text=[]
  Tables=[]
  for chunk in chunks:
    if "Table" in str(type(chunk)):
      Tables.append(chunk)
    if "CompositeElement" in str(type(chunk)):
      Text.append(chunk)
      meta_data=chunk.metadata.orig_elements

  return Text , Tables



# Store The Embedding Vectors into Chroma DB

In [5]:
from rank_bm25 import BM25Okapi
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

def create_vector_db(Text, Tables):
    client = chromadb.Client()
    text_embedding = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    text_collection = client.get_or_create_collection(
        name="text",
        embedding_function=text_embedding
    )

    text_strings = [str(chunk.text) for chunk in Text]
    table_strings = [str(tbl) for tbl in Tables]
    all_texts = text_strings + table_strings

    metadatas = []
    for chunk in Text:
        page_num = str(getattr(chunk.metadata, "page_number", "Unknown"))
        metadatas.append({"type": "text", "page": page_num})

    for tbl in Tables:
        page_num = str(getattr(tbl.metadata, "page_number", "Unknown"))
        metadatas.append({"type": "table", "page": page_num})

    text_collection.add(
        ids=[f"chunk_{i}" for i in range(len(all_texts))],
        documents=all_texts,
        metadatas=metadatas
    )


    return text_collection, all_texts, metadatas


# Preparing Prompt for LLM

In [6]:


def asking_question_hybrid(query, text_collection, all_texts, all_metadatas, llm):


    vector_results = text_collection.query(
        query_texts=[query],
        n_results=5
    )
    vector_docs = vector_results["documents"][0]
    vector_metas = vector_results["metadatas"][0]


    tokenized_corpus = [doc.lower().split(" ") for doc in all_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = query.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)


    top_bm25_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:5]


    combined_docs = []
    combined_metas = []
    seen = set()

    for doc, meta in zip(vector_docs + [all_texts[i] for i in top_bm25_indices],
                         vector_metas + [all_metadatas[i] for i in top_bm25_indices]):
        if doc not in seen:
            seen.add(doc)
            combined_docs.append(doc)
            combined_metas.append(meta)

    final_docs = combined_docs[:12]
    final_metas = combined_metas[:12]

    print(f"\n--- Question: {query} ---")
    for i, (doc, meta) in enumerate(zip(final_docs, final_metas)):
        print(f"[Chunk {i+1} | Page: {meta.get('page', 'N/A')} | Type: {meta.get('type')}] -> {doc}")
    print("-" * 50)

    context = "\n".join(final_docs) if final_docs else ""

    prompt = f"""
# Persona
You are an expert Nokia optical transport systems engineer with 10 years of experience.

# Task
Answer the user's question strictly using ONLY the direct facts found in the provided context chunks. If the core information is missing, state clearly that the documents do not contain it.

# Context
Context chunks retrieved from Nokia technical documentation.

# Format
Provide your response in a concise, structured bulleted list (keep it under 150 words). Do not include introductory phrases.

# Tone
Professional, objective, precise, and straight to the point.

# Guidelines
1. Keep your answer extremely direct, concise, and straight to the point.
2. Answer ONLY what is asked in the question. Do NOT include extra details or conversational fillers.
3. If the core information is missing from the chunks, state that the documents do not contain it.
4. If the question has multiple parts, you are fully allowed and encouraged to connect information from different chunks to provide a complete answer.

Context:
{context}

Question: {query}
Answer:"""

    answer = llm.invoke(prompt).content
    print(f"Answer:\n{answer}\n")
    return answer

In [32]:
def asking_question(query, db, llm):

    text_results = db.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas"]
    )

    retrieved_texts = text_results.get("documents", [[]])[0]
    retrieved_metas = text_results.get("metadatas", [[]])[0]

    print(f"\n--- Question: {query} ---")
    for i, (doc, meta) in enumerate(zip(retrieved_texts, retrieved_metas)):
        print(f"[Chunk {i+1} | Page: {meta.get('page', 'N/A')} | Type: {meta.get('type')}] -> {doc[:150]}...")
    print("-" * 50)

    context = "\n".join(retrieved_texts) if retrieved_texts else ""

    prompt = f"""
# Persona
You are an expert Nokia optical transport systems engineer with 10 years of experience.

# Task
Answer the user's question strictly using ONLY the direct facts found in the provided context chunks. If the core information is missing, state clearly that the documents do not contain it.

# Context
Context chunks retrieved from Nokia technical documentation.

# Format
Provide your response in a concise, structured bulleted list (keep it under 150 words). Do not include introductory phrases.

# Tone
Professional, objective, precise, and straight to the point.

# Guidelines
1. Keep your answer extremely direct, concise, and straight to the point.
2. Answer ONLY what is asked in the question. Do NOT include extra details or conversational fillers.
3. If the core information is missing from the chunks, state that the documents do not contain it.
4. If the question has multiple parts, you are fully allowed and encouraged to connect information from different chunks to provide a complete answer.

Context:
{context}

Question: {query}
Answer:"""

    answer = llm.invoke(prompt).content
    print(f"Answer:\n{answer}\n")
    return answer

In [8]:
PDF_Path="/content/extracted_nokia_pages.pdf"

In [10]:
chunks=PDF_Splitter(PDF_Path)
text, tables = Extract_data(chunks)

preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

In [11]:

text_collection, all_texts, all_metadatas = create_vector_db(text, tables)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
import os
import getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

Enter your Google Gemini API key: ··········


In [13]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",  google_api_key=os.environ["GOOGLE_API_KEY"])

# Without Hybrid search

In [33]:
response = asking_question("How many slots does the 1830 PSS-8 shelf provide, and what is its rack-unit (RU) footprint?", text_collection, llm)


--- Question: How many slots does the 1830 PSS-8 shelf provide, and what is its rack-unit (RU) footprint? ---
[Chunk 1 | Page: 43 | Type: text] -> 2.4.1 Introduction

The 1830 PSS-16II shelf provides the second generation 16-slot SWDM platform in an 8-RU footprint (DC variant) or a 9-RU footprint...
[Chunk 2 | Page: 41 | Type: text] -> 2.3.3 Slot numbering

The layout of the 1830 PSS-16 shelf and physical slot numbering are shown in the following figure.

Figure 2-9 1830 PSS-16 shelf...
[Chunk 3 | Page: 27 | Type: text] -> 2.1.1 Overview

The 1830 PSS shelves provide the framework where WDM application NEs are constructed. The 1830 PSS shelves can be deployed as a single...
[Chunk 4 | Page: 44 | Type: text] -> 2.4.3 Slot numbering

The 1830 PSS-16II shelf is partitioned into the following slot types:

• 16 paired half height application cards (slots 3-10 and...
[Chunk 5 | Page: 35 | Type: text] -> Flex shelf (FLEXxxxx)

The Flex Shelf is a mounting kit with cover that accepts the rack

In [34]:
response = asking_question("What rack-unit footprint does the 1830 PSS-32 shelf have, and how many slots does it provide?", text_collection, llm)


--- Question: What rack-unit footprint does the 1830 PSS-32 shelf have, and how many slots does it provide? ---
[Chunk 1 | Page: 43 | Type: text] -> 2.4.1 Introduction

The 1830 PSS-16II shelf provides the second generation 16-slot SWDM platform in an 8-RU footprint (DC variant) or a 9-RU footprint...
[Chunk 2 | Page: 41 | Type: text] -> 2.3.3 Slot numbering

The layout of the 1830 PSS-16 shelf and physical slot numbering are shown in the following figure.

Figure 2-9 1830 PSS-16 shelf...
[Chunk 3 | Page: 27 | Type: text] -> 2.1.1 Overview

The 1830 PSS shelves provide the framework where WDM application NEs are constructed. The 1830 PSS shelves can be deployed as a single...
[Chunk 4 | Page: 52 | Type: text] -> 2.5.3 Slot numbering

The 1830 PSS-32 shelf is partitioned into the following slot types:

• 32 paired half height application cards, or 16 full heigh...
[Chunk 5 | Page: 50 | Type: text] -> 2.5.1 Introduction

The 1830 PSS-32 shelf provides a 32-slot high-capacity and high-sc

In [35]:
response = asking_question("What are the two software load-lines supported by the 1830 PSS system?", text_collection, llm)


--- Question: What are the two software load-lines supported by the 1830 PSS system? ---
[Chunk 1 | Page: 1 | Type: text] -> System concept Overview

Nokia 1830 PSS-8/16II/16/32/ PSI-4L/PSI-8L

1 System concept

1.1 Overview

1.1.1 Purpose

This section provides an overview ...
[Chunk 2 | Page: 14 | Type: text] -> 1.3.1 Key innovations of the 1830 PSS

The 1830 Photonic Service Switch (PSS) portfolio consists of platforms optimized for varying optical network de...
[Chunk 3 | Page: 94 | Type: text] -> 1830 TDMX/TDMXC drop shelf with single-node management

This feature supports the integration of an 1830 PSS-32/PSS-16/PSS-16II/PSS-8 shelf with a 164...
[Chunk 4 | Page: 9 | Type: text] -> optical line resources can perform auto power management for OT line ports on other NEs. General management functions continue to be performed by the ...
[Chunk 5 | Page: 37 | Type: text] -> 2.2.5 Backplane

The 1830 PSS-8 backplane delivers 100Gb/s of transmission bandwidth from each I/O slot to the 

In [36]:
response = asking_question("Which fan unit(s) are used on the 1830 PSS-16II shelf?", text_collection, llm)


--- Question: Which fan unit(s) are used on the 1830 PSS-16II shelf? ---
[Chunk 1 | Page: 101 | Type: text] -> Shelves and common equipment/cards Fan units PSS-16 Fan Unit (FAN16)

Nokia 1830 PSS-8/16II/16/32/ PSI-4L/PSI-8L

2.16 PSS-16 Fan Unit (FAN16)

2.16.1...
[Chunk 2 | Page: 104 | Type: text] -> The 1830 PSS-16II FAN has 10 FAN units. Each FAN unit is powered separately. The 1830 PSS- 16II FAN can support a M+1 FAN unit protection scheme. When...
[Chunk 3 | Page: 102 | Type: text] -> 2.16.4 Front view

Figure 2-32 1830 PSS-16 fan unit

2.16.5 Visual indications

For information on the LED on the front panel of the FAN unit, see 2.3...
[Chunk 4 | Page: 41 | Type: text] -> 2.3.3 Slot numbering

The layout of the 1830 PSS-16 shelf and physical slot numbering are shown in the following figure.

Figure 2-9 1830 PSS-16 shelf...
[Chunk 5 | Page: 96 | Type: text] -> 2.15.1 Introduction

In the 1830 PSS-8 the fan unit is located in slot 14 of the shelf. The 1830 PSS-8 fan unit contains t

In [37]:
response = asking_question("Which fan units are supported on the 1830 PSS-32 shelf?", text_collection, llm)


--- Question: Which fan units are supported on the 1830 PSS-32 shelf? ---
[Chunk 1 | Page: 104 | Type: text] -> The 1830 PSS-16II FAN has 10 FAN units. Each FAN unit is powered separately. The 1830 PSS- 16II FAN can support a M+1 FAN unit protection scheme. When...
[Chunk 2 | Page: 101 | Type: text] -> Shelves and common equipment/cards Fan units PSS-16 Fan Unit (FAN16)

Nokia 1830 PSS-8/16II/16/32/ PSI-4L/PSI-8L

2.16 PSS-16 Fan Unit (FAN16)

2.16.1...
[Chunk 3 | Page: 41 | Type: text] -> 2.3.3 Slot numbering

The layout of the 1830 PSS-16 shelf and physical slot numbering are shown in the following figure.

Figure 2-9 1830 PSS-16 shelf...
[Chunk 4 | Page: 47 | Type: text] -> Shelf overview

The 1830 PSS-16II AC shelf is a 9RU high shelf and can fit into 300 mm depth rack.

fi]

Note: Due to different height, specific insta...
[Chunk 5 | Page: 96 | Type: text] -> 2.15.1 Introduction

In the 1830 PSS-8 the fan unit is located in slot 14 of the shelf. The 1830 PSS-8 fan unit contains t

In [38]:
response = asking_question("Name the power filter cards supported on the 1830 PSS-8 shelf.", text_collection, llm)


--- Question: Name the power filter cards supported on the 1830 PSS-8 shelf. ---
[Chunk 1 | Page: 46 | Type: text] -> 2.4.7 1830 PSS-16II shelf with support for integrated AC or DC power supply options

The 1830 PSS-16II supports an integrated AC/DC power filter modul...
[Chunk 2 | Page: 39 | Type: text] -> Note:

1. For DC power configuration, the version B 8DC30 power filter shall be used (3KC48870AB).

2. 1830 PSS-8 system is in compliance with IEEE 16...
[Chunk 3 | Page: 119 | Type: text] -> 2.22.2 External interfaces

The power filters (PFs) for the 1830 PSS-32 and 1830 PSS-16 shelves have a single power cable connector on the faceplate. ...
[Chunk 4 | Page: 117 | Type: text] -> PF (16AC16) front view

The faceplate of PSS-16II power filter card (16AC16) is shown in the following figure:

Figure 2-40 16AC16 faceplate

tum on p...
[Chunk 5 | Page: 41 | Type: text] -> 2.3.3 Slot numbering

The layout of the 1830 PSS-16 shelf and physical slot numbering are shown in the following fi

In [39]:
response = asking_question("What is the required horizontal rack aperture for mounting a 1830 PSS-8 shelf, and which common aperture size is explicitly NOT supported?", text_collection, llm)


--- Question: What is the required horizontal rack aperture for mounting a 1830 PSS-8 shelf, and which common aperture size is explicitly NOT supported? ---
[Chunk 1 | Page: 36 | Type: text] -> 2.2.2 Rack mounting options

The following rack types are supported:

• 19-inch EIA rack

• ETSI rack

• 23-inch ANSI rack

Note: When installing on E...
[Chunk 2 | Page: 43 | Type: text] -> 2.4.1 Introduction

The 1830 PSS-16II shelf provides the second generation 16-slot SWDM platform in an 8-RU footprint (DC variant) or a 9-RU footprint...
[Chunk 3 | Page: 47 | Type: text] -> Shelf overview

The 1830 PSS-16II AC shelf is a 9RU high shelf and can fit into 300 mm depth rack.

fi]

Note: Due to different height, specific insta...
[Chunk 4 | Page: 35 | Type: text] -> Flex shelf (FLEXxxxx)

The Flex Shelf is a mounting kit with cover that accepts the rack-mountable SFD44, DCMSHFxx, ATTNHDRW, ITLB, and FST. It occupi...
[Chunk 5 | Page: 27 | Type: text] -> 2.1.1 Overview

The 1830 PSS shelves prov

In [40]:
response = asking_question("What is the maximum optical reach, in kilometers, of the 1830 PSS-8 shelf without amplification?", text_collection, llm)


--- Question: What is the maximum optical reach, in kilometers, of the 1830 PSS-8 shelf without amplification? ---
[Chunk 1 | Page: 10 | Type: text] -> Notes:

1. Refer 1830 Photonic Service Switch (PSS-8/PSS-16II/PSS-16/PSS-32/PSI-4L/PSI-8L) Release 23.6.0 Customer Release Notes for the recommendatio...
[Chunk 2 | Page: 47 | Type: text] -> Shelf overview

The 1830 PSS-16II AC shelf is a 9RU high shelf and can fit into 300 mm depth rack.

fi]

Note: Due to different height, specific insta...
[Chunk 3 | Page: 36 | Type: text] -> 2.2.2 Rack mounting options

The following rack types are supported:

• 19-inch EIA rack

• ETSI rack

• 23-inch ANSI rack

Note: When installing on E...
[Chunk 4 | Page: 34 | Type: text] -> 2.1.6 Controller redundancy

The 1830 Photonic Service Switch shelves may be equipped with redundant equipment controllers (EC). Extension shelves tha...
[Chunk 5 | Page: 1 | Type: text] -> System concept Overview

Nokia 1830 PSS-8/16II/16/32/ PSI-4L/PSI-8L

1 System concep

# With Hybrid search

In [23]:

response = asking_question_hybrid(
    "How many slots does the 1830 PSS-8 shelf provide, and what is its rack-unit (RU) footprint?",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question: How many slots does the 1830 PSS-8 shelf provide, and what is its rack-unit (RU) footprint? ---
[Chunk 1 | Page: 43 | Type: text] -> 2.4.1 Introduction

The 1830 PSS-16II shelf provides the second generation 16-slot SWDM platform in an 8-RU footprint (DC variant) or a 9-RU footprint (AC/DC variant), with additional distributed switching functionality. The 1830 PSS-16II can support up to 8 full-height or up to 16 half-height universal I/O cards when the slot is also equipped with a half-slot adapter. The shelf has two slots dedicated for Power Filter modules, and two slots dedicated for Integrated Shelf Controllers (EC).

For detailed specifications, such as physical design, weight and power consumption, refer to

Chapter 7, “Technical specifications”

Figure 2-10 1830 PSS-16II shelf

SYN XYXNY OX SNK ORR RE x) “SLGOCS

2.4.2 Rack mounting options

The following rack types are supported:

• 19-inch EIA rack

Release 23.6 June 2023 Issue 1

© 2023 Nokia. Nokia Confidential

In [24]:

response = asking_question_hybrid(
    "What rack-unit footprint does the 1830 PSS-32 shelf have, and how many slots does it provide?",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question: What rack-unit footprint does the 1830 PSS-32 shelf have, and how many slots does it provide? ---
[Chunk 1 | Page: 43 | Type: text] -> 2.4.1 Introduction

The 1830 PSS-16II shelf provides the second generation 16-slot SWDM platform in an 8-RU footprint (DC variant) or a 9-RU footprint (AC/DC variant), with additional distributed switching functionality. The 1830 PSS-16II can support up to 8 full-height or up to 16 half-height universal I/O cards when the slot is also equipped with a half-slot adapter. The shelf has two slots dedicated for Power Filter modules, and two slots dedicated for Integrated Shelf Controllers (EC).

For detailed specifications, such as physical design, weight and power consumption, refer to

Chapter 7, “Technical specifications”

Figure 2-10 1830 PSS-16II shelf

SYN XYXNY OX SNK ORR RE x) “SLGOCS

2.4.2 Rack mounting options

The following rack types are supported:

• 19-inch EIA rack

Release 23.6 June 2023 Issue 1

© 2023 Nokia. Nokia Confidenti

In [25]:

response = asking_question_hybrid(
    " What are the two software load-lines supported by the 1830 PSS system?",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question:  What are the two software load-lines supported by the 1830 PSS system? ---
[Chunk 1 | Page: 1 | Type: text] -> System concept Overview

Nokia 1830 PSS-8/16II/16/32/ PSI-4L/PSI-8L

1 System concept

1.1 Overview

1.1.1 Purpose

This section provides an overview of 1830 Photonic Service Switch (PSS) product family, describing the configurations, system profile and network solutions of the system.

This document focuses on the support of 1830 PSS-8, 1830 PSS-16, 1830 PSS-16II, 1830 PSS-32, 1830 PSI-4L and 1830 PSI-8L. The shelves 1830 PSS-36, 1830 PSS-64 and 1830 PSS-8x/12x/24x are contained in this document in some general descriptions, for example in configurations, as they are used to build cluster configurations. Refer to 1830 Photonic Service Switch (PSS-4) Release 23.6 Product Information and Planning Guide, 1830 Photonic Service Switch (PSS-36/PSS-64) Release 14.1 Product Information and Planning Guide and 1830 Photonic Service Switch (PSS-8x/ PSS-12x/PSS-24x) Relea

In [31]:

response = asking_question_hybrid(
    " Which fan units are supported on the 1830 PSS-32 shelf?",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question:  Which fan units are supported on the 1830 PSS-32 shelf? ---
[Chunk 1 | Page: 104 | Type: text] -> The 1830 PSS-16II FAN has 10 FAN units. Each FAN unit is powered separately. The 1830 PSS- 16II FAN can support a M+1 FAN unit protection scheme. When there is one FAN motor unit failure, the system can continue working without heat dissipation issue, however, all FAN units will run in the highest rotation speed. The noise level will reach its summit.

2.17.7 Low power fan variant (16FAN2C)

A low power consuming unit, 16FAN2C (PN 3KC49100AA) variant supports 20A input power (less than 800W when Vin is around -39V). It consumes less power than 16FAN2, therefore generates less airflow than 16FAN2.

2.17.8 Node configuration support

Node configuration to PSS-16II shelf with 16FAN2C:

• RA5P in the shelf : 2xAWBILA + 2xRA5P

• OTDRWB in the shelf: 2xAWBILA+OTDRWB (optional)+RA5P (optional, RA5P number can be 0, 1, 2)

• RA5PB in the shelf : 2xAWBILA+OTDRWB (optional)+RA5PB (o

In [26]:

response = asking_question_hybrid(
    " Which fan unit(s) are used on the 1830 PSS-16II shelf?",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question:  Which fan unit(s) are used on the 1830 PSS-16II shelf? ---
[Chunk 1 | Page: 101 | Type: text] -> Shelves and common equipment/cards Fan units PSS-16 Fan Unit (FAN16)

Nokia 1830 PSS-8/16II/16/32/ PSI-4L/PSI-8L

2.16 PSS-16 Fan Unit (FAN16)

2.16.1 Introduction

The 1830 PSS-16 the fan unit is located at the bottom of the shelf. The fan unit contains 5 powerful FAN modules, each individually monitored and speed-controlled by network element (NE) software.

2.16.2 Fan unit replacement

When replacing the fan unit in an 1830 PSS-16 shelf, the only fan unit that supplies cooling air for the shelf is removed. The fan unit replacement is a simple procedure that can be accomplished in as little as 10 seconds. The fan unit replacement should be completed within 60 seconds.

Refer to the 1830 Photonic Service Switch (PSS) Release 23.6 Maintenance and Trouble-Clearing Guide for detailed instructions.

2.16.3 Air filter

Air for cooling the 1830 PSS-16 is drawn through fans at the

In [27]:

response = asking_question_hybrid(
    " Name the power filter cards supported on the 1830 PSS-8 shelf.",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question:  Name the power filter cards supported on the 1830 PSS-8 shelf. ---
[Chunk 1 | Page: 46 | Type: text] -> 2.4.7 1830 PSS-16II shelf with support for integrated AC or DC power supply options

The 1830 PSS-16II supports an integrated AC/DC power filter module. The AC power filters operate in a 1+1 protected configuration and the AC power should be greater than 2000W. Simplified timing is required. The 1830 PSS-16II AC shelf variant supports unrestricted configuration with AC power (up to 2.5kW). The maximum system power consumption is planned around 2460W to support the 8 OT slots with power consumption of maximum 240W per slot.

The PF-AC card consists of 2 independent boards: the power board for AC/DC and DC/DC conversion and LED control, the timing/control board for S-CRU and other control circuit. There is no direct electrical connection between them, but go through backplane. Physically they are bound together and logically NE SW shall treat it as an integrated card.



In [28]:

response = asking_question_hybrid(
    "What is the required horizontal rack aperture for mounting a 1830 PSS-8 shelf, and which common aperture size is explicitly NOT supported?",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question: What is the required horizontal rack aperture for mounting a 1830 PSS-8 shelf, and which common aperture size is explicitly NOT supported? ---
[Chunk 1 | Page: 36 | Type: text] -> 2.2.2 Rack mounting options

The following rack types are supported:

• 19-inch EIA rack

• ETSI rack

• 23-inch ANSI rack

Note: When installing on ETSI rack, air deflector 3KC50248AB can be used. For more details, refer 1830 Photonic Service Switch (PSS) Installation and System Turn-Up Guide.

Note: The horizontal aperture of the rack must be 450.85 mm (17.75 in). Racks with 444.5 mm (17.5 in) horizontal aperture are not supported.

82

© 2023 Nokia. Nokia Confidential Information Use subject to agreed restrictions on disclosure and use.

3KC-71311-QAAA-HQZZA

3KC-71311-QAAA-HQZZA

Release 23.6 June 2023 Issue 1

Shelves and common equipment/cards Shelves 1830 PSS-8 shelf

Nokia 1830 PSS-8/16II/16/32/ PSI-4L/PSI-8L

2.2.3 Slot numbering

The 1830 PSS-8 shelf is partitioned into the following 

In [29]:

response = asking_question_hybrid(
    "What is the maximum optical reach, in kilometers, of the 1830 PSS-8 shelf without amplification?",
    text_collection,
    all_texts,
    all_metadatas,
    llm
)


--- Question: What is the maximum optical reach, in kilometers, of the 1830 PSS-8 shelf without amplification? ---
[Chunk 1 | Page: 10 | Type: text] -> Notes:

1. Refer 1830 Photonic Service Switch (PSS-8/PSS-16II/PSS-16/PSS-32/PSI-4L/PSI-8L) Release 23.6.0 Customer Release Notes for the recommendation of the support in the release.

2. All network elements with involved 1830 PSS-24x shelves must have high-performance equipment controllers (32EC2) installed at the main shelf. It is in the customer's responsibility to enforce this.

3. The maximum number of 1830 PSS-8x shelves + 1830 PSS-12x shelves + 1830 PSS-24x shelves per NE must not exceed 8.

4. For multi-shelf configuration with PSS-32 main and PSS-32 extension (equipped with 32EC2) and PSS-16II extension, it is recommended to use 32EC2 in the PSS-32 main shelf.

5. The maximum number of 1830 PSS-8 shelves + 1830 PSS-16II shelves per NE must not exceed 8.

56

© 2023 Nokia. Nokia Confidential Information Use subject to agreed re